# XGBoost Average


In [1]:
# import libraries
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity as cosine
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import torch
pd.options.display.float_format = '{:,.2f}'.format

np.set_printoptions(threshold=np.inf)

In [2]:
#data from https://github.com/aliannejadi/istas

df=pd.read_json("data/istas.json")
df

,timestamp,UserID,Query,App,AppUsages
0,2018-04-14 13:28:32.655,54,"peoria, il post office, glen ave",google,"{'Duration': {'???': 1128776, 'android_system'..."
1,2018-04-14 19:06:05.105,54,"peoria, il weather",chrome,"{'Duration': {'???': 1128776, 'android_system'..."
10,2018-04-17 14:43:36.174,54,paypal/prepaid,google,"{'Duration': {'???': 538689, 'achievement': 39..."
100,2018-04-27 06:51:59.737,41,nose rod exam,puffin,"{'Duration': {'accessibility': 26381, 'all-in-..."
1000,2018-04-16 19:20:12.848,246,random alphanumeric generator,google,"{'Duration': {'ally_mobile': 50046, 'android_s..."
...,...,...,...,...,...
994,2018-04-16 05:41:03.138,126,hotel kidnapping at north spfld mo,chrome,"{'Duration': {'android_system': 63575, 'badgep..."
995,2018-04-16 14:41:01.883,126,why are the disney cartoons in spanish today,chrome,"{'Duration': {'android_system': 63575, 'badgep..."
997,2018-04-16 17:01:06.085,246,weather hope mills,google,"{'Duration': {'ally_mobile': 50046, 'android_s..."
998,2018-04-16 17:03:38.204,246,kakuriyo,chrome,"{'Duration': {'ally_mobile': 50046, 'android_s..."


In [3]:
from collections import Counter

user_app_counts = df.groupby(['UserID', 'App']).size().reset_index(name='count')
valid_user_apps = user_app_counts[user_app_counts['count'] > 1][['UserID', 'App']]
df_filtered = df.merge(valid_user_apps, on=['UserID', 'App'], how='inner')

df_filtered

,timestamp,UserID,Query,App,AppUsages
0,2018-04-14 13:28:32.655,54,"peoria, il post office, glen ave",google,"{'Duration': {'???': 1128776, 'android_system'..."
1,2018-04-17 14:43:36.174,54,paypal/prepaid,google,"{'Duration': {'???': 538689, 'achievement': 39..."
2,2018-04-17 20:21:09.314,54,comeuppance definition,google,"{'Duration': {'30_day_fitness_ch': 686217, '??..."
3,2018-04-19 02:55:55.770,54,paypal/prepaid,google,"{'Duration': {'achievement': 401511, 'android_..."
4,2018-04-14 20:37:49.631,54,sheridan nursery,google,"{'Duration': {'???': 1128776, 'android_system'..."
...,...,...,...,...,...
6251,2018-04-21 23:12:33.774,54,usps,yahoo_mail,"{'Duration': {'30_day_fitness_ch': 0, 'achieve..."
6252,2018-04-21 23:12:45.800,54,ipsy,yahoo_mail,"{'Duration': {'30_day_fitness_ch': 0, 'achieve..."
6253,2018-04-11 05:20:54.856,126,7 day forecast,weather_forecast,"{'Duration': {'???': 1346212, 'android_system'..."
6254,2018-04-11 15:12:47.533,126,april 11 forcast,weather_forecast,"{'Duration': {'???': 1346212, 'android_system'..."


In [4]:
df_filtered['App'] = df_filtered['App'].str.replace(r'[^\w\s]', ' ', regex=True)  
df_filtered['App'] = df_filtered['App'].str.replace(r'_', '')

apps = df_filtered['App'].unique()
print(apps)

['google' 'chrome' 'puffin' 'youtube' 'maps' 'messages' 'amazonshopping'
 'facebook' 'firefox' 'gmail' 'drive' 'myfitnesspal' 'samsunginternet'
 'googleplaystore' 'thepchapp' 'instagram' 'waze' 'memo' 'netflix' 'bing'
 'reddit' 'whatsapp' 'assistant' 'yelp' 'etsy' 'email' 'googleplaymusic'
 'cplusforcraigsl' 'inbox' 'navegador' 'configuración' 'telegram'
 'samsungnotes' 'messenger' 'spotify' 'pinterest' 'roku' 'photos'
 'contacts' 'settings' 'tripadvisor' 'calendar' 'bluemail' 'yahoomail'
 'hulu' 'fiosmobile' 'googleplaybooks' 'pandora' 'outlook' 'translate'
 'aol' 'radiojavan' 'phone' 'quora' 'mapquest' 'amazonmusic' 'imdb'
 'wikipedia' 'twitter' 'popcorntime' 'dropbox' 'walmart' 'sbbmobile'
 'fitbit' 'opera' 'atom' 'novalauncher' 'cakebrowser' 'brave' 'groupon'
 'snapchat' 'bbcnews' 'swagbucks' 'googleopinionre' 'sbtv' 'rolltheball'
 'calculator' 'toluna' 'receipthog' 'clock' 'bbciplayerradio' 'periscope'
 'cmcumobile' 'webvideocaster' 'slickdeals' 'robinhood' 'podcastaddict'
 'kitsu

In [5]:
import requests
import time
from bs4 import BeautifulSoup
from fake_useragent import UserAgent

In [7]:
#https://www.crummy.com/software/BeautifulSoup/bs4/doc/

def get_package_name(app_name):
    
    # build the search URL
    # try avoid getting banned by the server while performing consecutive requests, but should be fine for a small number of requests
    
    ua = UserAgent()
    
    headers = {'User-Agent':str(ua.chrome)}
    url = "https://play.google.com/store/search"
    params = {"q": app_name, "c": "apps"}

    response = requests.get(url, params=params, headers=headers)

    if response.status_code != 200:
        return None

    # get the link of the first result
    soup = BeautifulSoup(response.text, 'html.parser')
    app_link = soup.find('a', href=True, class_='Qfxief')  

    # since there are two different layouts for the search results, find in this class if the previous one is not found
    if not app_link:
        app_link = soup.find('a', href=True, class_='Si6A0c Gy4nib')

    # avoid getting banned by the server while performing consecutive requests
    time.sleep(1) 

    # get the package name 
    if app_link:
        return app_link['href'].split('id=')[1]  
    return None


def get_app_category(app_package_name):
    
    # build the URL for the app details page
    app_details_url = f"https://play.google.com/store/apps/details?id={app_package_name}"

    response = requests.get(app_details_url)

    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, 'html.parser')

    # look for the category tag
    category_tag = soup.find('a', class_='WpHeLc VfPpkd-mRLv6 VfPpkd-RLmnJb')

    time.sleep(1) 

    if category_tag and 'aria-label' in category_tag.attrs:
        return category_tag['aria-label']
    else:
        return None


for app_name in apps:   
   
    package_name = get_package_name(app_name)
    
    # if the package name is not found, mark the category as unknown
    category = get_app_category(package_name) if package_name else "unknown"


app_category_dict = {}

# go through each app name and get the package name
for app_name in apps:

    package_name = get_package_name(app_name)

  
    if package_name:
        category = get_app_category(package_name)
    else:
        category = "Unknown"

    app_category_dict[app_name] = category


    print(f"App: {app_name}, Category: {category}")


df_filtered['category'] = df_filtered['App'].map(app_category_dict)


df_filtered

App: google, Category: Communication
App: chrome, Category: Communication
App: puffin, Category: Tools
App: youtube, Category: Video Players & Editors
App: maps, Category: Travel & Local
App: messages, Category: Communication
App: amazonshopping, Category: Shopping
App: facebook, Category: Social
App: firefox, Category: Communication
App: gmail, Category: Communication
App: drive, Category: Productivity
App: myfitnesspal, Category: Health & Fitness
App: samsunginternet, Category: Communication
App: googleplaystore, Category: Entertainment
App: thepchapp, Category: Lifestyle
App: instagram, Category: Social
App: waze, Category: Maps & Navigation
App: memo, Category: Productivity
App: netflix, Category: Entertainment
App: bing, Category: Tools
App: reddit, Category: Social
App: whatsapp, Category: Communication
App: assistant, Category: Productivity
App: yelp, Category: Food & Drink
App: etsy, Category: Shopping
App: email, Category: Communication
App: googleplaymusic, Category: Music & 

,timestamp,UserID,Query,App,AppUsages,category
0,2018-04-14 13:28:32.655,54,"peoria, il post office, glen ave",google,"{'Duration': {'???': 1128776, 'android_system'...",Communication
1,2018-04-17 14:43:36.174,54,paypal/prepaid,google,"{'Duration': {'???': 538689, 'achievement': 39...",Communication
2,2018-04-17 20:21:09.314,54,comeuppance definition,google,"{'Duration': {'30_day_fitness_ch': 686217, '??...",Communication
3,2018-04-19 02:55:55.770,54,paypal/prepaid,google,"{'Duration': {'achievement': 401511, 'android_...",Communication
4,2018-04-14 20:37:49.631,54,sheridan nursery,google,"{'Duration': {'???': 1128776, 'android_system'...",Communication
...,...,...,...,...,...,...
6251,2018-04-21 23:12:33.774,54,usps,yahoomail,"{'Duration': {'30_day_fitness_ch': 0, 'achieve...",Communication
6252,2018-04-21 23:12:45.800,54,ipsy,yahoomail,"{'Duration': {'30_day_fitness_ch': 0, 'achieve...",Communication
6253,2018-04-11 05:20:54.856,126,7 day forecast,weatherforecast,"{'Duration': {'???': 1346212, 'android_system'...",Weather
6254,2018-04-11 15:12:47.533,126,april 11 forcast,weatherforecast,"{'Duration': {'???': 1346212, 'android_system'...",Weather


In [112]:
# remove the users that have less than 20 queries (to avoid train/test split errors).
user_sample_counts = df_filtered['UserID'].value_counts()
valid_users = user_sample_counts[user_sample_counts >= 20].index

df_filtered_valid = df_filtered[df_filtered['UserID'].isin(valid_users)]

df_filtered_valid = df_filtered_valid.dropna()

df_filtered_valid = df_filtered_valid.dropna(subset=['AppUsages'])

df_filtered_valid = df_filtered_valid[df_filtered_valid['AppUsages'].apply(
    lambda x: isinstance(x, dict) and 'Duration' in x and x['Duration'] is not None and x['Duration'] != ""
)]

app_counts_per_user = df_filtered_valid.groupby(['UserID', 'App']).size().reset_index(name='counts')
apps_to_keep = app_counts_per_user[app_counts_per_user['counts'] > 1]

df_filtered_valid = df_filtered_valid.merge(apps_to_keep[['UserID', 'App']], on=['UserID', 'App'], how='inner')


df_filtered_valid

,timestamp,UserID,Query,App,AppUsages,category
0,2018-04-14 13:28:32.655,54,"peoria, il post office, glen ave",google,"{'Duration': {'???': 1128776, 'android_system'...",Communication
1,2018-04-17 14:43:36.174,54,paypal/prepaid,google,"{'Duration': {'???': 538689, 'achievement': 39...",Communication
2,2018-04-17 20:21:09.314,54,comeuppance definition,google,"{'Duration': {'30_day_fitness_ch': 686217, '??...",Communication
3,2018-04-19 02:55:55.770,54,paypal/prepaid,google,"{'Duration': {'achievement': 401511, 'android_...",Communication
4,2018-04-14 20:37:49.631,54,sheridan nursery,google,"{'Duration': {'???': 1128776, 'android_system'...",Communication
...,...,...,...,...,...,...
5182,2018-04-21 23:12:33.774,54,usps,yahoomail,"{'Duration': {'30_day_fitness_ch': 0, 'achieve...",Communication
5183,2018-04-21 23:12:45.800,54,ipsy,yahoomail,"{'Duration': {'30_day_fitness_ch': 0, 'achieve...",Communication
5184,2018-04-11 05:20:54.856,126,7 day forecast,weatherforecast,"{'Duration': {'???': 1346212, 'android_system'...",Weather
5185,2018-04-11 15:12:47.533,126,april 11 forcast,weatherforecast,"{'Duration': {'???': 1346212, 'android_system'...",Weather


In [120]:
from transformers import BertTokenizer, BertModel

# Load BERT tokenizer and model
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


### Model Training

In [118]:
import torch
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

def custom_train_test_split(df_user, test_size=0.2, random_state=42):

    # custom split method to ensure each app appears at least once in the train and test sets
    

    apps = df_user['App'].unique()


    test_indices = []
    train_indices = []

    for app in apps:
        # get the indices of the app
        app_indices = df_user[df_user['App'] == app].index

        if len(app_indices) > 1:
            # if the app has more than one record, randomly select one for the test set
            test_index = np.random.choice(app_indices, 1, replace=False)[0]  # randomly get one for the test set
            test_indices.append(test_index)

            # the rest go to the training set
            remaining_indices = app_indices.drop(test_index)
            train_indices.extend(remaining_indices)
        else:
            # if the app has only one record, add it to the training set
            train_indices.extend(app_indices)

    # random shuffle the rest indices
    remaining_df = df_user.loc[train_indices]
    
    # make sure their is at least one record for each app in the test set
    num_classes = len(remaining_df['App'].unique())
    adjusted_test_size = max(test_size, num_classes / len(remaining_df))

    # check if all classes have at least two records
    if all(remaining_df['App'].value_counts() > 1):
        # if all the classes have at least two records, use stratify to ensure the same distribution in the test set
        X_train, X_test = train_test_split(
            remaining_df, test_size=adjusted_test_size, random_state=random_state, stratify=remaining_df['App']
        )
    else:
        X_train, X_test = train_test_split(
            remaining_df, test_size=adjusted_test_size, random_state=random_state
        )

    # add the test indices to the test set
    final_test = pd.concat([df_user.loc[test_indices], X_test])
    final_train = X_train

    # make sure the test set has the same classes as the training set
    train_classes = set(final_train['App'].unique())
    final_test = final_test[final_test['App'].isin(train_classes)]  # remove any classes not in the training set

    return final_train, final_test


def process_user_data(df_user, model, tokenizer, batch_size=32, n_components=0.95, num_clusters=10, test_size=0.2):

    # do data processing for each user
    
    # 1. One-Hot encoding for App categories and PCA reduction
    app_categories = df_user['category'].values.reshape(-1, 1)
    encoder = OneHotEncoder()
    app_category_encoded = encoder.fit_transform(app_categories).toarray()
    pca = PCA(n_components=n_components)
    category_reduced = pca.fit_transform(app_category_encoded)

    # 2. get the BERT embeddings for the queries and apps
    queries = df_user['Query'].tolist()
    app_names = df_user['App'].tolist()
    encoded_query = tokenizer.batch_encode_plus(queries, padding=True, truncation=True, return_tensors='pt', add_special_tokens=True)
    encoded_app = tokenizer.batch_encode_plus(app_names, padding=True, truncation=True, return_tensors='pt', add_special_tokens=True)
    
    query_input_ids = encoded_query['input_ids']
    query_attention_mask = encoded_query['attention_mask']
    app_input_ids = encoded_app['input_ids']
    app_attention_mask = encoded_app['attention_mask']
    
    query_embeddings = []
    app_embeddings = []
    
    for i in range(0, len(query_input_ids), batch_size):
        with torch.no_grad():
            outputs = model(query_input_ids[i:i+batch_size], attention_mask=query_attention_mask[i:i+batch_size])
        query_embeddings.append(outputs.last_hidden_state[:, 0, :])

    for i in range(0, len(app_input_ids), batch_size):
        with torch.no_grad():
            outputs = model(app_input_ids[i:i+batch_size], attention_mask=app_attention_mask[i:i+batch_size])
        app_embeddings.append(outputs.last_hidden_state[:, 0, :])

    CLS_query_embeddings = torch.cat(query_embeddings, dim=0).numpy()
    CLS_app_embeddings = torch.cat(app_embeddings, dim=0).numpy()

    # 3. PCA reduction for Query embeddings
    pca = PCA(n_components=n_components)
    query_reduced = pca.fit_transform(CLS_query_embeddings)

    # 4. KMeans clustering for Query embeddings
    kmeans = KMeans(n_clusters=num_clusters, random_state=42)
    cluster_labels = kmeans.fit_predict(query_reduced).reshape(-1, 1)

    # 5. flatten the app usage data and normalize it
    df_expanded = pd.json_normalize(df_user['AppUsages'].apply(lambda x: x.get('Duration') if isinstance(x, dict) else None))


    scaler = MinMaxScaler()

    if not df_expanded.empty and df_expanded.select_dtypes(include=[np.number]).shape[1] > 0:
        # combine and standardize the app usage data
        df_final = pd.concat([df_user.reset_index(drop=True), df_expanded.reset_index(drop=True)], axis=1).fillna(0).drop(columns=['AppUsages'])
        scaler = MinMaxScaler()
        df_final[df_expanded.columns] = scaler.fit_transform(df_final[df_expanded.columns])
        app_usage_features = df_final[df_expanded.columns].values
        app_usage_reduced = PCA(n_components=int(n_components)).fit_transform(app_usage_features)
    else:
        # if there is no app usage data, fill with zeros
        app_usage_reduced = np.zeros((df_user.shape[0], int(n_components)))

    # 6. combine all the features and normalize them
    all_features = np.concatenate([query_reduced, category_reduced, app_usage_reduced, cluster_labels], axis=1)
    all_features_normalized = scaler.fit_transform(all_features)

    # 7. split the data into training and testing sets
    X_train, X_test = custom_train_test_split(df_user[['App']].assign(features=list(all_features_normalized)), test_size=test_size)

    y_train = X_train['App']
    y_test = X_test['App']
    
    X_train = np.vstack(X_train['features'])
    X_test = np.vstack(X_test['features'])

    # 8. encode the labels
    le = LabelEncoder()
    le.fit(df_user['App'])  
    y_train = le.transform(y_train)  
    y_test = le.transform(y_test)    


    return X_train, X_test, y_train, y_test, cluster_labels, cluster_labels

def batch_process_users(df, model, tokenizer, batch_size=32, n_components=0.95, num_clusters=10, test_size=0.1):

    # train the model on all users
    user_results = {}
    unique_user_ids = df['UserID'].unique()

    for user_id in unique_user_ids:

        df_user = df[df['UserID'] == user_id]

        X_train, X_test, y_train, y_test, cluster_train, cluster_test = process_user_data(
            df_user, model, tokenizer, batch_size=batch_size, n_components=n_components, num_clusters=num_clusters, test_size=test_size
        )

        # save the results for each user for later use
        user_results[user_id] = {
            'X_train': X_train,
            'X_test': X_test,
            'y_train': y_train,
            'y_test': y_test,
            'cluster_train': cluster_train,
            'cluster_test': cluster_test
        }

    return user_results

user_results = batch_process_users(df_filtered_valid, model, tokenizer)

c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)
c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\sklearn\cluster\_kmeans.py:1416: FutureWarning: The default value of `n_init` will change fr

### Evaluations

In [119]:
import xgboost as xgb
from sklearn.metrics import accuracy_score
import numpy as np
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

#Top-K Accuracy
def top_k_accuracy(y_true, y_prob, k=3):
    top_k_preds = np.argsort(y_prob, axis=1)[:, -k:]  
    correct = np.sum([1 if y_true[i] in top_k_preds[i] else 0 for i in range(len(y_true))])
    return correct / len(y_true)

#MRR(Mean Reciprocal Rank)
def mean_reciprocal_rank(y_true, y_prob):
    ranks = []
    for i, true_app in enumerate(y_true):
        sorted_pred = np.argsort(y_prob[i])[::-1]  
        rank = np.where(sorted_pred == true_app)[0][0] + 1  
        ranks.append(1 / rank)
    return np.mean(ranks)

#P@1(Precision at 1)
def precision_at_1(y_true, y_prob):
    top_1_preds = np.argmax(y_prob, axis=1)  
    return np.mean(top_1_preds == y_true)


def train_and_evaluate_user(X_train, y_train, X_test, y_test):

    model_tree = xgb.XGBClassifier(
        use_label_encoder=False,
        eval_metric='mlogloss',
        learning_rate=0.08,
        max_depth=4,
        n_estimators=200
    )

    model_tree.fit(X_train, y_train)

    y_pred = model_tree.predict(X_test)
    y_prob = model_tree.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    top_3_acc = top_k_accuracy(y_test, y_prob, k=3)
    mrr_score = mean_reciprocal_rank(y_test, y_prob)
    p_at_1 = precision_at_1(y_test, y_prob)

    return accuracy, top_3_acc, mrr_score, p_at_1

# do the evaluation for each user
def batch_train_and_evaluate(user_results):
    total_accuracy, total_top_3_acc, total_mrr, total_p_at_1 = 0, 0, 0, 0
    num_users = len(user_results)
    low_accuracy_users = []

    for user_id, data in user_results.items():
        X_train, X_test, y_train, y_test = data['X_train'], data['X_test'], data['y_train'], data['y_test']

        le = LabelEncoder()
        y_combined = np.concatenate((y_train, y_test))  
        le.fit(y_combined)
        y_train = le.transform(y_train)
        y_test = le.transform(y_test)

        accuracy, top_3_acc, mrr_score, p_at_1 = train_and_evaluate_user(X_train, y_train, X_test, y_test)

        total_accuracy += accuracy
        total_top_3_acc += top_3_acc
        total_mrr += mrr_score
        total_p_at_1 += p_at_1

        # 记录 accuracy < 60% 的用户
        if accuracy < 0.60:
            low_accuracy_users.append(user_id)

        print(f"User {user_id} - Accuracy: {accuracy:.4f}, Top-3 Accuracy: {top_3_acc:.4f}, MRR: {mrr_score:.4f}, P@1: {p_at_1:.4f}")

    avg_accuracy = total_accuracy / num_users
    avg_top_3_acc = total_top_3_acc / num_users
    avg_mrr = total_mrr / num_users
    avg_p_at_1 = total_p_at_1 / num_users

    print(f"\nAverage Metrics across {num_users} users:")
    print(f"Average Accuracy: {avg_accuracy:.4f}")
    print(f"Average Top-3 Accuracy: {avg_top_3_acc:.4f}")
    print(f"Average MRR: {avg_mrr:.4f}")
    print(f"Average P@1: {avg_p_at_1:.4f}")

    print(f"\nUsers with accuracy < 60%: {low_accuracy_users}")

    return low_accuracy_users


low_accuracy_users = batch_train_and_evaluate(user_results)


c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


User 54 - Accuracy: 0.5294, Top-3 Accuracy: 0.7647, MRR: 0.6947, P@1: 0.5294
User 41 - Accuracy: 0.8571, Top-3 Accuracy: 1.0000, MRR: 0.9048, P@1: 0.8571


c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


User 128 - Accuracy: 0.6667, Top-3 Accuracy: 0.7778, MRR: 0.7667, P@1: 0.6667
User 101 - Accuracy: 0.6667, Top-3 Accuracy: 1.0000, MRR: 0.8333, P@1: 0.6667
User 241 - Accuracy: 0.8750, Top-3 Accuracy: 1.0000, MRR: 0.9375, P@1: 0.8750
User 86 - Accuracy: 0.8333, Top-3 Accuracy: 1.0000, MRR: 0.8889, P@1: 0.8333


c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:29] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:30] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\toby

...

User 85 - Accuracy: 0.6923, Top-3 Accuracy: 0.9231, MRR: 0.8141, P@1: 0.6923
User 158 - Accuracy: 0.5000, Top-3 Accuracy: 0.6364, MRR: 0.6384, P@1: 0.5000


c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:42] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)
c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:42] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


User 90 - Accuracy: 1.0000, Top-3 Accuracy: 1.0000, MRR: 1.0000, P@1: 1.0000


c:\Users\tobys\miniconda3\envs\nlp\lib\site-packages\xgboost\core.py:158: UserWarning: [21:33:42] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0015a694724fa8361-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


User 218 - Accuracy: 0.7255, Top-3 Accuracy: 0.8627, MRR: 0.8040, P@1: 0.7255

Average Metrics across 70 users:
Average Accuracy: 0.7067
Average Top-3 Accuracy: 0.9024
Average MRR: 0.8143
Average P@1: 0.7067

Users with accuracy < 60%: [54, 87, 105, 8, 10, 169, 254, 155, 138, 220, 117, 229, 198, 150, 237, 219, 141, 158]
